# Orchestration & Lakeflow Pipelines

**Module 5 | Fundamentals Training**

Workflows (Databricks Jobs) allows you to automate and schedule notebooks and scripts. We focus on **creating simple tasks in the GUI**, parameterization via Widgets, and scheduling recurring jobs. At the end we cover **Lakeflow Pipelines** — a declarative approach to building data pipelines.


## Learning Objectives

After completing this module you will be able to:

- **Explain** what Databricks Workflows are and when to use them
- **Describe** the difference between All-Purpose clusters and Job clusters for scheduled tasks
- **Create** a simple single-task Job in the Workflows GUI
- **Add parameters** to a notebook using `dbutils.widgets`
- **Schedule** a recurring job using CRON or a simple time trigger
- **Monitor** a job run and understand run status, logs, and error messages

> **Focus:** GUI-first approach — no Databricks Asset Bundles (DABs) or CI/CD in this module. That comes in the next training level (Explorer).


### Task Types in Lakeflow Jobs

Overview of all available task types in Lakeflow Jobs and when to use each one.

| Task Type | Description | Use Case |
|-----------|-------------|----------|
| Notebook | Run a Databricks notebook | ETL logic, ML training |
| Pipeline | Run a Lakeflow Declarative Pipeline | Streaming/batch pipelines |
| Python Script | Run a Python file | Utility scripts |
| SQL | Run a SQL query | DDL, reporting queries |
| JAR | Run a Java/Scala JAR | Legacy Spark jobs |
| Spark Submit | Submit a Spark application | Custom Spark apps |
| If/Else Condition | Branch based on condition | Conditional workflows |
| For Each | Iterate over a list | Parameterized batch runs |

> **Tip:** Repair Run re-executes only failed tasks and their downstream dependencies — skipping already-successful tasks. If/Else and For Each enable conditional and iterative workflows.

## Preparing Notebooks for Job

Below are 3 simple notebooks that we'll use in the demo.

**Instructions**: 
1. Create folder `/Workspace/Users/<your-email>/jobs_demo/`
2. Copy each of the following code snippets to a separate notebook

---

### Task 1: Validate Source

Validates row count against `min_rows` threshold. Returns status + count via `dbutils.notebook.exit()`.

In [0]:
# TASK 1: Validate Source Data
# Copy this code to notebook: task_01_validate

# Parameters from Job
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("min_rows", "100")

source_table = dbutils.widgets.get("source_table")
min_rows = int(dbutils.widgets.get("min_rows"))

# Validation
df = spark.table(source_table)
row_count = df.count()

if row_count < min_rows:
    raise Exception(f"Validation FAILED: {row_count} rows < {min_rows} minimum")

# Return result to next task
import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "source_table": source_table,
    "row_count": row_count
}))

### Task 2: Transform Data

Reads previous task result via `taskValues`, applies transformations (duration, cost per mile). Returns row count.

In [0]:
# TASK 2: Transform Data
# Copy this code to notebook: task_02_transform

from pyspark.sql.functions import *
import json

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("run_date", "")

source_table = dbutils.widgets.get("source_table")
run_date = dbutils.widgets.get("run_date") or str(current_date())

# Get result from previous task (optional)
try:
    prev_result = dbutils.jobs.taskValues.get(
        taskKey="validate",
        key="returnValue",
        default="{}"
    )
    prev_data = json.loads(prev_result)
    print(f"Previous task result: {prev_data}")
except:
    print("Running standalone (no previous task)")

# Transformation
print(f"Transforming: {source_table}")

df = spark.table(source_table)

df_transformed = (
    df
    .withColumn("trip_duration_minutes", 
                round((col("tpep_dropoff_datetime").cast("long") - 
                       col("tpep_pickup_datetime").cast("long")) / 60, 2))
    .withColumn("cost_per_mile", 
                when(col("trip_distance") > 0, 
                     round(col("fare_amount") / col("trip_distance"), 2))
                .otherwise(0))
    .withColumn("processing_date", lit(run_date))
)

row_count = df_transformed.count()
print(f"Transformed {row_count} rows")

df_transformed.select(
    "trip_distance", "fare_amount", "trip_duration_minutes", "cost_per_mile"
).show(5)

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "rows_transformed": row_count
}))

### Task 3: Generate Report

Aggregates metrics (total trips, revenue, avg fare/distance). Prints summary report.

In [0]:
# TASK 3: Generate Report
# Copy this code to notebook: task_03_report

from pyspark.sql.functions import *
import json
from datetime import datetime

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")

source_table = dbutils.widgets.get("source_table")

# Aggregations
df = spark.table(source_table)

report = df.agg(
    count("*").alias("total_trips"),
    round(sum("fare_amount"), 2).alias("total_revenue"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(max("fare_amount"), 2).alias("max_fare")
).collect()[0]

# Display report
print("\n" + "="*50)
print("DAILY REPORT")
print("="*50)
print(f"Avg Fare:       ${report.avg_fare:.2f}")
print(f"Avg Distance:   {report.avg_distance:.2f} miles")
print(f"Max Fare:       ${report.max_fare:.2f}")
print("="*50)
print(f"Generated at:   {datetime.now()}")
print("="*50 + "\n")

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "total_trips": report.total_trips,
    "total_revenue": float(report.total_revenue)
}))

### [UI DEMO] Creating Multi-task Job

**Step 1: Create new Job**
- [ ] Workflows → Create Job
- [ ] Name: `Demo_ETL_Pipeline_<your_name>`

<img src="../../assets/images/93c107ca21a54aab98249cf47db0337d.png" width="800">


- [ ] Cluster Job: Create new cluster job

<img src="../../assets/images/a967557a143a40c0ac7ed26ce469866a.png" width="800">

**Step 2: Add Task 1 (Validate)**
- [ ] Task name: `validate`
- [ ] Type: Notebook
- [ ] Path: `/Workspace/.../task_01_validate`
- [ ] Cluster: Serverless or new Job Cluster
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

<img src="../../assets/images/214b868309344df3a6e81f3cc2a84c13.png" width="800">

**Step 3: Add Task 2 (Transform)**
- [ ] Task name: `transform`
- [ ] Depends on: `validate`
- [ ] Path: `/Workspace/.../task_02_transform`
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

**Step 4: Add Task 3 (Report)**
- [ ] Task name: `report`
- [ ] Depends on: `transform`
- [ ] Path: `/Workspace/.../task_03_report`

<img src="../../assets/images/a3cc387f44de4247bf275bdbd38efb84.png" width="800">

**Step 5: Run Job**
- [ ] Run now
- [ ] Show: DAG visualization
- [ ] Show: Task logs and output

---

## [UI DEMO] Options, Retry and Alerting

**Task-level:** Timeout, Retries (count + delay)  
**Job-level:** Max concurrent runs, Job timeout  
**Notifications:** Email (on failure/success), Webhooks (Slack/Teams via Destinations)

| Scenario | Retry? | Why |
|----------|--------|-----|
| Network timeout | Yes | Transient error |
| API rate limit | Yes | Transient error |
| Data quality issue | No | Retry won't fix data |
| Code bug | No | Retry won't fix code |

---

## Monitoring via System Tables

Key table: `system.lakeflow.job_run_timeline` — contains run history, duration, result state.

---

In [0]:
spark.sql("""
    SELECT 
        *
    FROM system.lakeflow.job_run_timeline
    ORDER BY 1 DESC
    LIMIT 20
""").display()

In [0]:
spark.sql("""
    SELECT 
        DATE(period_start_time) as run_date,
        run_name as job_name,
        ROUND(
            AVG(
                (UNIX_TIMESTAMP(period_end_time) - UNIX_TIMESTAMP(period_start_time)) / 60
            ), 1
        ) as avg_duration_min,
        COUNT(*) as runs
    FROM system.lakeflow.job_run_timeline
    WHERE period_start_time >= current_date() - INTERVAL 14 DAYS
        AND result_state = 'SUCCESS'
    GROUP BY run_date, job_name
    ORDER BY run_date DESC
""").display()

---
## Lakeflow Pipelines (preview)

> **Databricks Lakeflow Pipelines** is a new, declarative layer on top of Databricks Workflows introduced in 2024. It allows you to define data pipelines as a **task graph** with automatic dependency management between tables.

### How do Lakeflow Pipelines differ from Workflows (Jobs)?

| Feature | Databricks Workflows (Jobs) | Lakeflow Pipelines |
|---------|---------------------------|-------------------|
| Definition style | Imperative (step by step) | Declarative (describe what, not how) |
| Granularity | Task (notebook, script, DLT) | Table (each table = node in the graph) |
| Dependencies | Manually defined between tasks | Automatically detected from `LIVE.` prefixes |
| Error handling | Retry at task level | Automatic retry per table with checkpoint |
| Integration | General (Python, SQL, Jar, dbt) | Dedicated to Delta Live Tables |
| Monitoring | Job Run UI | Pipeline UI with data flow visualization |

### Lakeflow Architecture — conceptual view

![2a391f6919494d21925176522c18ec0e.png](../../assets/images/2a391f6919494d21925176522c18ec0e.png "2a391f6919494d21925176522c18ec0e.png")

### Object types in Lakeflow / Delta Live Tables:

| Object | SQL Syntax | What it is |
|--------|-----------|-----------|
| `STREAMING TABLE` | `CREATE OR REFRESH STREAMING TABLE` | Incremental ingest — reads only new data |
| `MATERIALIZED VIEW` | `CREATE OR REFRESH MATERIALIZED VIEW` | Refreshed view — delta from previous state |
| `VIEW` | `CREATE VIEW` | Temporary view — not stored as a table |

### Minimal working pipeline (SQL)

```sql
-- Bronze: read new files from landing zone
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT * FROM STREAM('landing_orders');

-- Silver: deduplication + type guard
CREATE OR REFRESH MATERIALIZED VIEW silver_orders
AS SELECT DISTINCT order_id, customer_id, CAST(amount AS DOUBLE) AS amount
FROM bronze_orders
WHERE order_id IS NOT NULL;

-- Gold: aggregation for BI
CREATE OR REFRESH MATERIALIZED VIEW gold_orders_by_customer
AS SELECT customer_id, SUM(amount) AS total_spent, COUNT(*) AS order_count
FROM silver_orders
GROUP BY customer_id;
```

### How to start a Lakeflow Pipeline:
1. Left menu: **Jobs & Pipeline ** (or **New → Pipeline**)
2. **Create pipeline** → enter a name
3. Select Source (e.g., SQL or Python notebooks with `dlt.` decorators)
4. Select mode: **Triggered** (one-time refresh) or **Continuous** (streaming)
5. Click **Start** — the pipeline will build the table graph and begin processing

> **When to use Lakeflow instead of a regular Workflow?**  
> Lakeflow = data pipelines (Bronze → Silver → Gold) with automatic dependency management.  
> Workflow = any task orchestration (ML training, reports, backfill, ETL between systems).


## Summary

| Topic | Key Concept | Keywords |
|-------|-------------|----------|
| **Multi-task Jobs** | DAG workflow, task dependencies | Task types, Repair Runs |
| **Triggers** | Scheduled (CRON), File arrival, Continuous | `0 0 2 * * ?` |
| **Options** | Timeout, Retry, Max concurrent runs | Transient vs permanent errors |
| **Alerting** | Email, Webhooks (Slack/Teams) | Notification destinations |
| **Parameters** | Widgets, dynamic values, taskValues | `dbutils.widgets`, `dbutils.jobs.taskValues` |
| **Monitoring** | System tables, success rate, duration | `system.lakeflow.job_run_timeline` |
| **Lakeflow Pipelines** | Declarative Delta Live Tables, STREAMING TABLE, MATERIALIZED VIEW | `LIVE.`, Triggered vs Continuous |

---

> ← [Delta Lake](04_delta_fundamentals.ipynb) | [Unity Catalog →](06_unity_catalog.ipynb)
